# Validate a credit-risk score with FORML

This notebook shows the workflow an ML engineer could use before deployment:

1. train and serialize a small affine scoring model;
2. define behavioral requirements in a `.forml` policy;
3. run `verify(...)` without manipulating IR2 or Z3 directly;
4. inspect the rich HTML report;
5. replay a counterexample on the original sklearn model;
6. export stable JSON and standalone HTML artifacts.

The example uses **transformed numerical features** (`debt_ratio` and `late_payments`). Encoding preprocessing pipelines is deliberately outside the current V1 scope.

In [ ]:
from fractions import Fraction
from pathlib import Path
from tempfile import TemporaryDirectory

import joblib
import pandas as pd
from sklearn.linear_model import LinearRegression

from dsl.backends import VerificationStatus
from dsl.runtime import verify


def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "dsl").is_dir() and (candidate / "demo").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from inside the FORML repository")


ROOT = find_repository_root(Path.cwd())
POLICY_PATH = ROOT / "demo" / "notebooks" / "credit_risk_policy.forml"
workspace_handle = TemporaryDirectory(prefix="forml-credit-risk-notebook-")
WORKSPACE = Path(workspace_handle.name)
WORKSPACE

## 1. Build a small reference model

The synthetic target is intentionally affine so that the current Z3 profile can encode it:

\[
\text{risk\_score} = 0.6 \times \text{debt\_ratio}
                  + 0.08 \times \text{late\_payments}
                  + 0.1
\]

The score is illustrative rather than a calibrated probability.

In [ ]:
rows = []
for debt_ratio in (0.0, 0.25, 0.5, 0.75, 1.0):
    for late_payments in (0.0, 1.0, 2.0, 3.0, 4.0, 5.0):
        rows.append(
            {
                "debt_ratio": debt_ratio,
                "late_payments": late_payments,
                "risk_score": 0.6 * debt_ratio + 0.08 * late_payments + 0.1,
            }
        )

reference = pd.DataFrame(rows)
features = ["debt_ratio", "late_payments"]
model = LinearRegression().fit(reference[features], reference["risk_score"])

model_path = WORKSPACE / "credit_risk.joblib"
dataset_path = WORKSPACE / "credit_risk_reference.csv"
joblib.dump(model, model_path)
reference.to_csv(dataset_path, index=False)

pd.DataFrame(
    {
        "feature": features,
        "coefficient": model.coef_,
    }
).assign(intercept=model.intercept_)

## 2. Review the behavioral policy

The policy proves a global upper bound, deliberately exposes a stricter rule that has a counterexample, and asks for an applicant reaching the manual-review threshold.

In [ ]:
print(POLICY_PATH.read_text(encoding="utf-8"))

## 3. Verify the serialized model

`verify(...)` performs model introspection, semantic validation, IR construction, capability routing, Z3 execution and report construction. In Jupyter, returning the session as the final expression invokes its `_repr_html_()` method.

In [ ]:
session = verify(
    POLICY_PATH,
    model=model_path,
    dataset=dataset_path,
)
session

In [ ]:
pd.DataFrame(
    [
        {
            "property": report.property_index + 1,
            "type": report.property_type.value,
            "status": report.status.value,
            "specification": report.specification,
        }
        for report in session.reports
    ]
)

## 4. Replay the counterexample on sklearn

A formal counterexample is most useful when it can be checked against the original model artifact. The following cell converts exact Z3 rational values to Python floats, reconstructs the input row and compares both outputs.

In [ ]:
def solver_number_to_float(value: object) -> float:
    text = str(value)
    return float(Fraction(text)) if "/" in text else float(text)


counterexample = next(
    report
    for report in session.reports
    if report.status is VerificationStatus.COUNTEREXAMPLE
)

counterexample_inputs = {
    assignment.display_name.split(".", 1)[1]: solver_number_to_float(assignment.value)
    for assignment in counterexample.inputs
}
input_row = pd.DataFrame([counterexample_inputs], columns=features)

sklearn_prediction = float(model.predict(input_row)[0])
forml_prediction = solver_number_to_float(counterexample.outputs[0].value)

pd.DataFrame(
    [
        {
            **counterexample_inputs,
            "sklearn_prediction": sklearn_prediction,
            "forml_prediction": forml_prediction,
            "absolute_difference": abs(sklearn_prediction - forml_prediction),
        }
    ]
)

## 5. Export review artifacts

The JSON contract is versioned for CI or downstream automation. The HTML file is self-contained and can be attached to a model-review ticket without requiring Jupyter.

In [ ]:
json_report = session.write_json(
    WORKSPACE / "reports" / "credit-risk-report.json"
)
html_report = session.write_html(
    WORKSPACE / "reports" / "credit-risk-report.html"
)

{
    "json": json_report,
    "html": html_report,
    "ci_exit_code": session.exit_code,
}

A non-zero `session.exit_code` is suitable for a CI gate. This notebook intentionally contains a counterexample, so the overall session exits with code `1` even though that counterexample is expected for demonstration purposes.